In [1]:
import os
import math
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import pandas as pd
from matplotlib.gridspec import GridSpec

In [3]:
csv_folder = "/hd2/marcos/research/repos/pig-segmentation-distill/dataset-scenarios/descriptions"
def load_dictionaries():
    """
    Reads the CSV files and converts them into Python dictionaries 
    for fast lookup.
    """
    dicts = {}
    
    files = {
        'SID': 'SID_list.csv',
        'IID': 'IID_list.csv',
        'EID': 'EID_list.csv',
        'AID': 'AID_list.csv'
    }

    for key, filename in files.items():
        path = os.path.join(csv_folder, filename)
        if os.path.exists(path):
            try:
                df = pd.read_csv(path)
                dicts[key] = pd.Series(df.iloc[:, 1].values, index=df.iloc[:, 0].astype(str)).to_dict()
            except Exception as e:
                print(f"Warning: Could not read {filename}. Error: {e}")
                dicts[key] = {}
        else:
            print(f"Warning: {filename} not found in {csv_folder}")
            dicts[key] = {}
            
    return dicts

In [14]:
def decode_filename(filename, lookup_dicts):
    """
    Parses '1010s1120s2001-2s5300-1-1.jpg' and returns a readable string.
    Structure assumed: SID s IID s EID s AID-FN
    """
    clean_name = os.path.splitext(filename)[0]
    
    parts = clean_name.split('s')
    
    if len(parts) < 4:
        return filename
    
    sid_id = parts[0]
    iid_id = parts[1]
    eid_id = parts[2]
    
    aid_part_split = parts[3].rsplit('-', 1)
    aid_id = aid_part_split[0] 
    
    sid_desc = lookup_dicts['SID'].get(sid_id, sid_id)
    iid_desc = lookup_dicts['IID'].get(iid_id, iid_id)
    eid_desc = lookup_dicts['EID'].get(eid_id, eid_id)
    aid_desc = lookup_dicts['AID'].get(aid_id, aid_id)
    
    label = f"{filename}\n{sid_desc}\n{iid_desc}\n{eid_desc}\n{aid_desc}"
    return label

In [15]:
def plot_grid_with_labels(image_dir):
    lookups = load_dictionaries()
    
    all_files = sorted([f for f in os.listdir(image_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
    
    scenario_map = {}
    for img in all_files:
        scenario_prefix = img.rsplit('-', 1)[0]
        if scenario_prefix not in scenario_map:
            scenario_map[scenario_prefix] = img

    scenarios = list(scenario_map.values())
    total = len(scenarios)
    
    if total == 0:
        print("No images found.")
        return

    cols = 5
    rows = math.ceil(total / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(18, 4 * rows))
    axes = axes.flatten()

    for i, ax in enumerate(axes):
        if i < total:
            img_filename = scenarios[i]
            img_path = os.path.join(image_dir, img_filename)
            
            readable_title = decode_filename(img_filename, lookups)
            
            try:
                img_data = mpimg.imread(img_path)
                ax.imshow(img_data)
                ax.set_title(readable_title, fontsize=9, color='darkblue') 
            except:
                ax.text(0.5, 0.5, "Error", ha='center')
            
            ax.axis('off')
        else:
            ax.axis('off')

    plt.tight_layout()
    plt.show()

In [41]:
def count_unique_scenarios(image_dir):
    images = os.listdir(image_dir)
    scenarios = list(set("s".join(img.split("s")[0:2]) for img in images))
    return scenarios

In [39]:
def plot_specific_group(image_dir, group_id):
    """
    Plots all images in a directory that belong to a specific group ID.
    
    Args:
        image_dir (str): Path to the folder containing images.
        group_id (str): The prefix/id of the group to plot (e.g., "scenario_01").
    """
    try:
        # Get all image files
        all_files = sorted([f for f in os.listdir(image_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
    except FileNotFoundError:
        print(f"Error: Directory '{image_dir}' not found.")
        return

    # Filter: Keep only images that start with the group_id
    # We add a '-' check to ensure we don't match "scenario_10" when looking for "scenario_1"
    # If your naming convention is different, you can remove the + '-' part.
    target_images = [f for f in all_files if f.startswith(group_id)]
    
    total_images = len(target_images)
    
    if total_images == 0:
        print(f"No images found for group: {group_id}")
        return

    # Grid settings
    cols = 5
    rows = math.ceil(total_images / cols)

    # Handle the figure size dynamically based on rows
    fig, axes = plt.subplots(rows, cols, figsize=(15, 3 * rows))
    
    # Ensure axes is always iterable (handle case of 1 row or 1 image)
    if total_images == 1:
        axes = [axes]
    elif rows > 1 or cols > 1:
        axes = axes.flatten()

    for i, ax in enumerate(axes):
        if i < total_images:
            img_name = target_images[i]
            img_path = os.path.join(image_dir, img_name)
            
            try:
                img_data = mpimg.imread(img_path)
                ax.imshow(img_data)
                # Clean up title: remove extension and group_id for cleaner look
                display_title = img_name.replace(group_id, '').strip('-_ .')
                ax.set_title(display_title if display_title else img_name, fontsize=8)
            except Exception as e:
                ax.text(0.5, 0.5, "Error loading", ha='center')
            
            ax.axis('off')
        else:
            # Turn off empty subplots in the grid
            ax.axis('off')

    plt.suptitle(f"Group: {group_id}", fontsize=14)
    plt.tight_layout()
    plt.show()

In [21]:
def get_group_title(sid, iid, lookups):
    """Returns a readable group title: 'Nursery | RGB Camera'"""
    s_desc = lookups.get('SID', {}).get(sid, sid)
    i_desc = lookups.get('IID', {}).get(iid, iid)
    return f"SUBJECT: {s_desc}  |  CAMERA: {i_desc}"

In [22]:
def get_image_label(filename, lookups):
    """Returns the subtitle for individual images (Environment | Activity)"""
    parts = filename.replace('.jpg', '').replace('.png', '').split('s')
    if len(parts) < 4: return filename
    
    eid = lookups.get('EID', {}).get(parts[2], parts[2])
    aid_raw = parts[3].rsplit('-', 1)[0]
    aid = lookups.get('AID', {}).get(aid_raw, aid_raw)
    
    return f"{eid}\n{aid}"

### SAM3-PigLife 

#### Test subset

In [46]:
path = "/hd2/marcos/research/repos/pig-segmentation-distill/teacher/test/images"
scenarios = count_unique_scenarios(path)

#### Validation subset

In [ ]:
path = "/hd2/marcos/research/repos/pig-segmentation-distill/data/SAM3_PigLife/val/images"
# count_unique_scenarios(path)
# plot_grid_with_labels(path)

#### Train subset

In [ ]:
path = "/hd2/marcos/research/repos/pig-segmentation-distill/data/SAM3_PigLife/train/images"
# count_unique_scenarios(path)
# plot_grid_with_labels(path)

In [ ]:
import os
import math
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import random

IMAGE_DIR = '/hd2/marcos/research/repos/pig-segmentation-distill/teacher/test/images'
REPORT_DIR = '/hd2/marcos/research/repos/pig-segmentation-distill/dataset-scenarios'
REPORT_FILE = 'scenario_report.md'

TARGET_GROUPS = [
    ["1050s1132a1110s3003-3s5001"],
    ["1050s1132a1110s3003-5s5001"],
    ["1050s1132a1110s3004","1050s1132a1112"],
    ["1010s1120", "1010s1121"],
    ["1060s1112", "1060a1000s1110"],
    ["1020s1120", "1020s1220"],
    ["1040s1132"],
    ["1020s1121", "1020s1221"]
]

def generate_report_pipeline():
    os.makedirs(REPORT_DIR, exist_ok=True)
    images_out_dir = os.path.join(REPORT_DIR, 'plots')
    os.makedirs(images_out_dir, exist_ok=True)
    
    md_path = os.path.join(REPORT_DIR, REPORT_FILE)
    with open(md_path, 'w') as f:
        f.write(f"# Scenario Visual Report\n\nGenerated pipeline run.\n\n---\n\n")

    try:
        all_files = sorted([f for f in os.listdir(IMAGE_DIR) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
    except FileNotFoundError:
        print(f"Error: Source directory '{IMAGE_DIR}' not found.")
        return

    for index, group_ids in enumerate(TARGET_GROUPS, 1):
        print(f"Processing Group {index}: {group_ids}...")
        
        matched_images = []
        for f in all_files:
            if any(f.startswith(gid) for gid in group_ids):
                matched_images.append(f)
        
        total_found_count = len(matched_images)
        
        if total_found_count == 0:
            print(f"  -> No images found. Skipping.")
            continue

        if total_found_count > 3:
            matched_images = random.sample(matched_images, 3)
            matched_images = sorted(matched_images)
        
        create_and_save_plot(index, group_ids, matched_images, total_found_count, images_out_dir, md_path)

def create_and_save_plot(index, group_ids, images, total_count, output_dir, md_file_path):
    images_to_show = len(images)
    cols = 3 
    rows = 1 

    fig, axes = plt.subplots(rows, cols, figsize=(15, 5))
    
    if images_to_show == 1:
        axes = [axes]
    elif isinstance(axes, plt.Axes): 
        axes = [axes]
    elif rows > 1 or cols > 1:
        axes = axes.flatten()

    for i in range(cols):
        if i < len(axes): 
            ax = axes[i]
            if i < images_to_show:
                img_path = os.path.join(IMAGE_DIR, images[i])
                try:
                    img_data = mpimg.imread(img_path)
                    ax.imshow(img_data)
                    ax.set_title(images[i], fontsize=9)
                except Exception:
                    ax.text(0.5, 0.5, "Error loading", ha='center')
                ax.axis('off')
            else:
                ax.axis('off')

    group_title = ", ".join(group_ids)
    
    plt.suptitle(f"Group #{index}: {group_title} (Found: {total_count})", fontsize=12)
    plt.tight_layout()
    
    plot_filename = f"group_{index}_random_summary.png"
    save_path = os.path.join(output_dir, plot_filename)
    plt.savefig(save_path)
    plt.close(fig)

    rel_path = f"./plots/{plot_filename}"
    
    with open(md_file_path, 'a') as f:
        f.write(f"### Group {index}\n")
        f.write(f"**Filter IDs:** `{group_title}`\n")
        f.write(f"**Total Images Available:** {total_count}\n\n")
        f.write(f"![Group {index}]({rel_path})\n\n")
        f.write("---\n\n")
        
    print(f"  -> Saved random selection (found {total_count}) to {save_path}")

if __name__ == "__main__":
    generate_report_pipeline()